<a href="https://colab.research.google.com/github/deepugangadhar46/ecommerce-db/blob/main/Ecommerce_PySpark_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pyspark psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 62.8 MB/s eta 0:00:00


In [2]:
import os, json

os.makedirs("data/sample_clickstream", exist_ok=True)
os.makedirs("data/sample_store_sales", exist_ok=True)

# Write sample clickstream JSON
clickstream_data = """{"session_id": "sess_101", "user_id": "U1001", "page_url": "/home", "device": "mobile", "timestamp": "2026-10-09 10:05:00"}
{"session_id": "sess_101", "user_id": "U1001", "page_url": "/product/shoes", "device": "mobile", "timestamp": "2026-10-09 10:12:00"}
{"session_id": "sess_102", "user_id": "U1002", "page_url": "/product/laptop", "device": "desktop", "timestamp": "2026-10-09 10:15:30"}
{"session_id": "sess_103", "user_id": "U1003", "page_url": "/cart", "device": "mobile", "timestamp": "2026-10-09 10:20:00"}
{"session_id": "sess_104", "user_id": null, "page_url": "/product/watch", "device": "desktop", "timestamp": "2026-10-09 10:25:00"}
{"session_id": "sess_105", "user_id": "", "page_url": "/offers", "device": "tablet", "timestamp": "2026-10-09 10:30:00"}
{"session_id": "sess_106", "user_id": "U1004", "page_url": "/product/phone", "device": "mobile", "timestamp": "2026-10-09 10:45:00"}
{"session_id": "sess_107", "user_id": "U1005", "page_url": "/checkout", "device": "desktop", "timestamp": "2026-10-09 11:05:00"}
{"session_id": "sess_108", "user_id": "U1001", "page_url": "/home", "device": "mobile", "timestamp": "2026-10-09 11:10:00"}
{"session_id": "sess_109", "user_id": "U1006", "page_url": "/product/bag", "device": "desktop", "timestamp": "2026-10-09 11:15:00"}
{"session_id": "sess_110", "user_id": "U1007", "page_url": "/product/headphones", "device": "mobile", "timestamp": "2026-10-09 11:22:00"}
{"session_id": "sess_111", "user_id": "U1008", "page_url": "/faq", "device": "desktop", "timestamp": "2026-10-09 11:35:00"}
{"session_id": "sess_112", "user_id": null, "page_url": "/contact", "device": "mobile", "timestamp": "2026-10-09 11:40:00"}
{"session_id": "sess_113", "user_id": "U1009", "page_url": "/product/shoes", "device": "tablet", "timestamp": "2026-10-09 12:05:00"}
{"session_id": "sess_114", "user_id": "U1010", "page_url": "/home", "device": "desktop", "timestamp": "2026-10-09 12:15:00"}
{"session_id": "sess_115", "user_id": "U1002", "page_url": "/checkout", "device": "desktop", "timestamp": "2026-10-09 12:30:00"}
{"session_id": "sess_116", "user_id": "U1011", "page_url": "/product/shirt", "device": "mobile", "timestamp": "2026-10-09 12:45:00"}
{"session_id": "sess_117", "user_id": "U1012", "page_url": "/about", "device": "desktop", "timestamp": "2026-10-09 12:55:00"}"""

with open("data/sample_clickstream/clickstream_data.json", "w") as f:
    f.write(clickstream_data)

# Write sample store sales CSV
store_sales_data = """transaction_id,user_id,store_id,amount,payment_method,timestamp
TXN_5001,U1001,STR_BLR_01,120.50,Credit_Card,2026-10-09 10:25:00
TXN_5002,U1002,STR_BLR_02,899.00,UPI,2026-10-09 10:45:00
TXN_5003,U1003,STR_BLR_01,45.00,Cash,2026-10-09 10:50:00
TXN_5004,,STR_BLR_03,32.00,UPI,2026-10-09 10:55:00
TXN_5005,U1004,STR_BLR_02,650.00,Debit_Card,2026-10-09 11:10:00
TXN_5006,U1005,STR_BLR_01,150.00,Credit_Card,2026-10-09 11:20:00
TXN_5007,U1001,STR_BLR_01,80.00,UPI,2026-10-09 11:45:00
TXN_5008,U1007,STR_BLR_03,-25.00,Cash,2026-10-09 11:50:00
TXN_5009,U9999,STR_BLR_02,210.00,UPI,2026-10-09 11:55:00
TXN_5010,U1009,STR_BLR_01,95.00,Debit_Card,2026-10-09 12:15:00
TXN_5011,U1010,STR_BLR_02,15.50,Cash,2026-10-09 12:25:00
TXN_5012,U1002,STR_BLR_02,110.00,Credit_Card,2026-10-09 12:40:00
TXN_5013,U1011,STR_BLR_03,450.00,UPI,2026-10-09 12:50:00
TXN_5014,,STR_BLR_01,75.00,Credit_Card,2026-10-09 12:55:00
TXN_5015,U1012,STR_BLR_01,0.00,Cash,2026-10-09 12:58:00"""

with open("data/sample_store_sales/store_sales_data.csv", "w") as f:
    f.write(store_sales_data)

print("Sample data files created successfully!")

Sample data files created successfully!


In [3]:
DB_CONFIG = {
    "host": "ep-sparkling-lake-b5z0l2e2-pooler.c-7.us-east-2.aws.neon.tech",       # e.g., ep-sparkling-xxx.us-east-2.aws.neon.tech
    "port": "5432",
    "dbname": "neondb",
    "user": "neondb_owner",
    "password": "npg_1qGNX3WUQEwM",
    "sslmode": "require"
}

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql import functions as F
import psycopg2
from psycopg2.extras import execute_values

# Initialize PySpark Session
spark = SparkSession.builder \
    .appName("EcommerceSalesPipeline") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# --- MEMBER 1: INGESTION & QUALITY CONTROL ---
clickstream_schema = StructType([
    StructField("session_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("page_url", StringType(), True),
    StructField("device", StringType(), True),
    StructField("timestamp", StringType(), True)
])

sales_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("timestamp", StringType(), True)
])

# Read and validate Clickstream
raw_click = spark.read.schema(clickstream_schema).json("data/sample_clickstream/clickstream_data.json")
df_click = raw_click.withColumn("event_time", F.to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss"))
clean_click = df_click.filter(F.col("user_id").isNotNull() & (F.trim(F.col("user_id")) != "") & F.col("event_time").isNotNull())
bad_click = df_click.filter(F.col("user_id").isNull() | (F.trim(F.col("user_id")) == "") | F.col("event_time").isNull()) \
    .withColumn("source_file", F.lit("clickstream_data.json")) \
    .withColumn("error_reason", F.lit("Missing or empty user_id")) \
    .withColumn("raw_payload", F.to_json(F.struct([F.col(c) for c in raw_click.columns])))

# Read and validate Sales
raw_sales = spark.read.option("header", "true").schema(sales_schema).csv("data/sample_store_sales/store_sales_data.csv")
df_sales = raw_sales.withColumn("txn_time", F.to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss"))
clean_sales = df_sales.filter(F.col("user_id").isNotNull() & (F.trim(F.col("user_id")) != "") & (F.col("amount") > 0) & F.col("txn_time").isNotNull())
bad_sales = df_sales.filter(F.col("user_id").isNull() | (F.trim(F.col("user_id")) == "") | (F.col("amount") <= 0) | F.col("txn_time").isNull()) \
    .withColumn("source_file", F.lit("store_sales_data.csv")) \
    .withColumn("error_reason", F.lit("Null user_id or negative/zero purchase amount")) \
    .withColumn("raw_payload", F.to_json(F.struct([F.col(c) for c in raw_sales.columns])))

print(f"[Member 1] Clean Clickstream: {clean_click.count()} | Quarantined: {bad_click.count()}")
print(f"[Member 1] Clean Store Sales: {clean_sales.count()} | Quarantined: {bad_sales.count()}")

# --- MEMBER 2: TRANSFORMATION & AGGREGATIONS ---
# Broadcast join (store_sales broadcasted to prevent expensive shuffling)
joined_df = clean_click.join(F.broadcast(clean_sales), on="user_id", how="inner")

fact_df = joined_df.groupBy(
    F.window(F.col("txn_time"), "1 hour").alias("time_window"),
    F.col("store_id")
).agg(
    F.countDistinct("session_id").alias("active_sessions"),
    F.round(F.sum("amount"), 2).alias("total_sales_amount"),
    F.countDistinct("transaction_id").alias("total_transactions")
).select(
    F.col("time_window.start").alias("date_hour"),
    F.col("store_id"),
    F.col("active_sessions"),
    F.col("total_sales_amount"),
    F.col("total_transactions")
)

print("\n[Member 2] Aggregated Results:")
fact_df.show(truncate=False)

# --- MEMBER 3: DATABASE LOADING ---
conn = psycopg2.connect(
    host=DB_CONFIG["host"],
    port=DB_CONFIG["port"],
    dbname=DB_CONFIG["dbname"],
    user=DB_CONFIG["user"],
    password=DB_CONFIG["password"],
    sslmode=DB_CONFIG["sslmode"]
)
cur = conn.cursor()

# 1. Load Fact Records
fact_records = [
    (row["date_hour"].strftime("%Y-%m-%d %H:%M:%S"), row["store_id"], int(row["active_sessions"]), float(row["total_sales_amount"]), int(row["total_transactions"]))
    for row in fact_df.collect()
]
fact_query = """
    INSERT INTO fact_hourly_sales_sessions (date_hour, store_id, active_sessions, total_sales_amount, total_transactions)
    VALUES %s
    ON CONFLICT (date_hour, store_id)
    DO UPDATE SET
        active_sessions = EXCLUDED.active_sessions,
        total_sales_amount = EXCLUDED.total_sales_amount,
        total_transactions = EXCLUDED.total_transactions;
"""
execute_values(cur, fact_query, fact_records)

# 2. Load Quarantine Records
combined_bad = bad_click.select("source_file", "raw_payload", "error_reason").union(
    bad_sales.select("source_file", "raw_payload", "error_reason")
).collect()
bad_records = [(r["source_file"], r["raw_payload"], r["error_reason"]) for r in combined_bad]
quarantine_query = "INSERT INTO quarantine_records (source_file, raw_payload, error_reason) VALUES %s;"
execute_values(cur, quarantine_query, bad_records)

conn.commit()
cur.close()
conn.close()

print(f"[Member 3] Successfully loaded {len(fact_records)} facts and {len(bad_records)} quarantined records into Neon PostgreSQL!")

[Member 1] Clean Clickstream: 15 | Quarantined: 3
[Member 1] Clean Store Sales: 11 | Quarantined: 4

[Member 2] Aggregated Results:
+-------------------+----------+---------------+------------------+------------------+
|date_hour          |store_id  |active_sessions|total_sales_amount|total_transactions|
+-------------------+----------+---------------+------------------+------------------+
|2026-10-09 11:00:00|STR_BLR_01|3              |390.0             |2                 |
|2026-10-09 12:00:00|STR_BLR_01|1              |95.0              |1                 |
|2026-10-09 10:00:00|STR_BLR_01|3              |406.5             |2                 |
|2026-10-09 12:00:00|STR_BLR_02|3              |235.5             |2                 |
|2026-10-09 10:00:00|STR_BLR_02|2              |1798.0            |1                 |
|2026-10-09 11:00:00|STR_BLR_02|1              |650.0             |1                 |
|2026-10-09 12:00:00|STR_BLR_03|1              |450.0             |1                 